# Cuantización: cómo cabe un modelo gigante en tu portátil

**Facsímil 3 · Arquitecturas y modelos** — capítulos 6 y 7
(modelos abiertos; inferencia optimizada, *edge AI* y hardware).

Un modelo de 7.000 millones de parámetros en *float32* ocupa unos 28 GB solo de pesos: no cabe en una
tarjeta gráfica modesta ni en tu portátil. Y sin embargo hoy corres modelos así en casa. ¿El truco?
La **cuantización**: guardar cada peso con menos bits. En este cuaderno la haces a mano, mides las dos
caras del trato —cuánta **memoria** ahorras y cuánto **error** introduces— y descubres por qué un
formato como `Q4` (los que ves en los modelos GGUF) vive justo en el filo de lo aceptable. No es magia:
es redondear con cabeza.

### Qué vas a aprender
- Por qué los modelos ocupan tanto y qué significa guardar un peso en *float32* frente a *int8*.
- El proceso de cuantización, paso a paso: encontrar la **escala**, **redondear** y volver.
- A medir el **error** que introduce (y a *verlo* en un histograma) y por qué casi siempre «cuela».
- Qué pasa con **menos bits** (4, 2) y con **valores atípicos**, los dos enemigos de la cuantización.
- Dos defensas reales: cuantizar **por filas** y usar **escala más desplazamiento** (asimétrica).
- Cuánto se acumula el error al **multiplicar** y cómo dimensionar un modelo real en memoria.

### Cuánto cuesta
Unos 16 minutos. CPU, sin claves. Lee cada celda de texto antes de correr la de código.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Simulamos una capa: una matriz de pesos en float32 (lo normal en un modelo).
pesos = np.random.randn(1024, 1024).astype(np.float32)
print("Matriz de pesos:", pesos.shape, "| tipo:", pesos.dtype)
print(f"Memoria en float32: {pesos.nbytes/1e6:.2f} MB  (4 bytes por numero)")
print(f"Un modelo de 7.000 millones de pesos en float32 ocuparia ~{7e9*4/1e9:.0f} GB. Por eso importa.")


### Una analogía cotidiana: la regla con menos marcas

Piensa en medir una mesa con dos reglas. Una tiene marcas cada milímetro (muchos valores posibles, como
*float32*); la otra solo cada centímetro (pocos valores, como *int8*). Con la regla de centímetros vas
más rápido y la apuntas con menos cifras, pero cada medida la **redondeas** a la marca más cercana: ahí
nace el error. Si encima usas una regla pensada para medir un campo de fútbol para medir un lápiz (una
escala demasiado grande, estirada por un valor enorme), las marcas te quedan tan separadas que el lápiz
mide «cero coma algo» y pierdes todo el detalle. Cuantizar es justo eso: elegir cuántas marcas tiene tu
regla y dónde las pones. Todo lo que sigue es ponerle números a esta intuición.


## 1. Cuantizar a 8 bits: redondear a una rejilla

En *float32* cada número usa 32 bits y puede tomar casi cualquier valor. En *int8* solo hay **256
valores** posibles (de -128 a 127). La idea de la cuantización:

1. Encontrar el peso de **mayor magnitud** (define el rango).
2. Dividir ese rango en escalones: la **escala** = (máximo) / 127.
3. Guardar cada peso como el **escalón entero más cercano** (esto es lo que ocupa solo 8 bits).
4. Para *usar* el peso, multiplicarlo de vuelta por la escala (*descuantizar*).

Es un redondeo, ida y vuelta. Lo que se guarda son los enteros y un único número (la escala) por
matriz.


In [ ]:
def cuantiza(w, bits=8):
    nivel = 2**(bits-1) - 1                 # 127 para 8 bits
    escala = np.abs(w).max() / nivel        # un solo numero por matriz
    q = np.round(w / escala).astype(np.int64)
    q = np.clip(q, -nivel, nivel)
    return q, escala

q, escala = cuantiza(pesos, bits=8)
recuperado = q.astype(np.float32) * escala

print(f"Pesos en 8 bits -> memoria: {q.astype(np.int8).nbytes/1e6:.2f} MB")
print(f"Reduccion: {pesos.nbytes / q.astype(np.int8).nbytes:.0f} veces mas pequeno (de 32 a 8 bits)\n")
print("Ejemplo (5 pesos):")
print("  original :", np.round(pesos[0,:5], 4))
print("  recuperado:", np.round(recuperado[0,:5], 4))


### Mirar un solo peso de cerca

Para que «redondear a la rejilla» deje de ser abstracto, sigamos a **un único peso** por todo el viaje:
su valor original, el escalón entero al que cae, y el valor que recuperamos al descuantizar. La
diferencia entre el primero y el último es el error de cuantización de ese peso, y nunca puede ser mayor
que media escala.


In [ ]:
w0 = float(pesos[0, 0])
escalon = int(np.round(w0 / escala))
vuelta = escalon * escala
print(f"Peso original           : {w0:.6f}")
print(f"Dividido por la escala  : {w0/escala:.3f}  -> redondea al escalon entero {escalon}")
print(f"Al descuantizar (escalon * escala): {vuelta:.6f}")
print(f"Error de este peso      : {abs(w0 - vuelta):.6f}")
print(f"Media escala (error maximo posible): {escala/2:.6f}  <- el error nunca pasa de aqui")


## 2. ¿Cuánto error metimos? Ponle número y dibújalo

Comparamos los pesos originales con los recuperados. El error existe (hemos redondeado), pero conviene
ponerle número y compararlo con lo que ahorramos. Una medida útil es la **relación señal/ruido** (SNR)
en decibelios: cuánta «señal» (el peso) hay frente al «ruido» (el error de redondeo). Cuanto más alta,
mejor. Además dibujamos el histograma del error para ver que es pequeño y está repartido.


In [ ]:
def snr_db(original, recuperado):
    senal = np.sqrt((original**2).mean())
    ruido = np.sqrt(((original - recuperado)**2).mean())
    return 20*np.log10(senal/ruido)

error = np.abs(pesos - recuperado)
print(f"Error medio por peso:  {error.mean():.6f}")
print(f"Error maximo por peso: {error.max():.6f}")
print(f"Senal/ruido (8 bits):  {snr_db(pesos, recuperado):.1f} dB (cuanto mas alto, mejor)")
print("\nResumen del trato: 4x menos memoria a cambio de un error minusculo en cada peso.")

plt.figure(figsize=(7, 3.4))
plt.hist((pesos - recuperado).ravel(), bins=60, color="steelblue")
plt.xlabel("error por peso (original - recuperado)"); plt.ylabel("nº de pesos")
plt.title("El error de cuantizar a 8 bits es pequeno y esta repartido")
plt.tight_layout(); plt.show()


## 3. Menos bits, más ahorro... y más error

8 bits no es el límite. Los formatos modernos bajan a 4 bits (`Q4`) o incluso menos, ahorrando aún más
memoria. Pero el error crece: con menos escalones, el redondeo es más brusco. Medimos el trato a 8, 4 y
2 bits para ver dónde deja de compensar.


In [ ]:
print("bits | valores posibles | memoria relativa | señal/ruido")
for bits in [8, 4, 2]:
    qb, esb = cuantiza(pesos, bits=bits)
    rec = qb.astype(np.float32) * esb
    print(f"  {bits}  |       {2**bits:>4}        |      {bits/32*100:>3.0f}%       |   {snr_db(pesos, rec):>5.1f} dB")
print("\nDe 8 a 4 bits: la mitad de memoria, pero el error sube. A 2 bits ya se degrada mucho.")
print("Ese filo (4 bits) es justo donde viven los modelos GGUF 'Q4' que ves para correr en casa.")


### Verlo: la rejilla se vuelve gruesa

¿Por qué menos bits = más error? Porque hay menos escalones para repartir el mismo rango, así que cada
escalón es más ancho y el redondeo, más brusco. Lo dibujamos: ordenamos unos pocos pesos reales y los
superponemos con su versión cuantizada a 8 y a 2 bits. A 2 bits la curva se convierte en una escalera
de poquísimos peldaños.


In [ ]:
muestra = np.sort(pesos[0, :40])
plt.figure(figsize=(7.5, 3.6))
plt.plot(muestra, "o-", color="black", label="original (float32)", markersize=3)
for bits, marca in [(8, "s--"), (2, "^:")]:
    qb, esb = cuantiza(muestra, bits=bits)
    plt.plot(qb.astype(np.float32) * esb, marca, label=f"{bits} bits", markersize=4)
plt.xlabel("peso (ordenados)"); plt.ylabel("valor")
plt.title("Menos bits, rejilla mas gruesa: la curva se vuelve escalera")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("A 8 bits casi se calca el original; a 2 bits solo quedan unos pocos peldanos.")


## 4. El enemigo nº1: los valores atípicos

La cuantización tiene un talón de Aquiles. La escala la fija el peso de **mayor magnitud**. Si un único
peso es un *outlier* enorme, estira la escala para todos: los escalones se vuelven gruesos y *todos* los
demás pesos (los normales) se cuantizan peor. Lo provocamos metiendo un solo valor atípico y vemos cómo
empeora el error del resto.


In [ ]:
pesos_out = pesos.copy()
pesos_out[0, 0] = 60.0          # un solo outlier gigante
q1, e1 = cuantiza(pesos, bits=8);        rec1 = q1.astype(np.float32)*e1
q2, e2 = cuantiza(pesos_out, bits=8);    rec2 = q2.astype(np.float32)*e2
# error solo en los pesos NORMALES (excluyendo el outlier)
err_sin = np.abs(pesos[1:] - rec1[1:]).mean()
err_con = np.abs(pesos_out[1:] - rec2[1:]).mean()
print(f"Escala SIN outlier: {e1:.4f}   |  Escala CON outlier: {e2:.4f}  (se ha estirado x{e2/e1:.0f})")
print(f"Error medio de los pesos normales SIN outlier: {err_sin:.6f}")
print(f"Error medio de los pesos normales CON outlier: {err_con:.6f}")
print(f"\nUn solo peso atipico ha empeorado el error de TODOS los demas x{err_con/err_sin:.1f}.")
print("Por eso las tecnicas buenas tratan los outliers aparte.")


### No es cuántos atípicos, es cuán grande es el peor

Aquí hay un detalle que sorprende. Como la escala es `máximo / 127`, **solo manda el peso más grande**:
meter más atípicos del mismo tamaño no cambia nada, pero uno solo más grande arrastra a todos. Medimos
las dos cosas: primero subimos la **magnitud** del atípico (el daño crece), y luego comprobamos que
añadir **más** atípicos del mismo tamaño deja el error igual.


In [ ]:
def error_normales(valor_outlier):
    w = pesos.copy()
    w[0, 0] = valor_outlier
    qd, ed = cuantiza(w, bits=8)
    rec = qd.astype(np.float32) * ed
    err = np.abs(w.ravel()[1:] - rec.ravel()[1:]).mean()   # todos menos el outlier [0,0]
    return err, ed

base, _ = error_normales(float(pesos[0, 0]))               # sin outlier (valor normal)
print(f"{'magnitud del peor':>17} | {'escala':>8} | {'error pesos normales':>20} | factor")
for valor in [3.0, 10.0, 30.0, 60.0, 150.0]:
    err, esc = error_normales(valor)
    print(f"{valor:>17.0f} | {esc:>8.4f} | {err:>20.6f} | x{err/base:.1f}")
print("\nCuanto mayor es el peor atipico, mas se estira la escala y peor van TODOS los demas.\n")

# Ahora: mismo tamano (60), pero metiendo 1, 5 y 20 atipicos
def error_con_n_outliers(cuantos, valor=60.0):
    w = pesos.copy().ravel(); w[:cuantos] = valor; w = w.reshape(pesos.shape)
    qd, ed = cuantiza(w, bits=8); rec = qd.astype(np.float32) * ed
    return np.abs(w.ravel()[cuantos:] - rec.ravel()[cuantos:]).mean()
for cuantos in [1, 5, 20]:
    print(f"  {cuantos:>2} atipicos de magnitud 60 -> error de los normales: {error_con_n_outliers(cuantos):.6f}")
print("Identico: anadir mas atipicos del mismo tamano no empeora nada. Manda solo el mayor.")


## 5. La defensa nº1: cuantizar por filas

Una mejora barata: en vez de una escala única para toda la matriz, usar **una escala por fila**. Así un
*outlier* en una fila solo afecta a esa fila, no a las demás. Cuesta un poco más de memoria (varias
escalas en vez de una), pero reduce el error. Lo comprobamos sobre la matriz con *outlier*.


In [ ]:
def cuantiza_por_filas(w, bits=8):
    nivel = 2**(bits-1) - 1
    escala = np.abs(w).max(axis=1, keepdims=True) / nivel     # una escala por fila
    q = np.clip(np.round(w / escala), -nivel, nivel)
    return q * escala

rec_global = cuantiza(pesos_out, 8)[0].astype(np.float32) * cuantiza(pesos_out, 8)[1]
rec_filas  = cuantiza_por_filas(pesos_out, 8)
print(f"Error medio (escala global): {np.abs(pesos_out - rec_global).mean():.6f}")
print(f"Error medio (escala por fila): {np.abs(pesos_out - rec_filas).mean():.6f}")
print("\nAislar el outlier en su fila protege al resto. Mas escalas, mas precision, algo mas de memoria.")


## 6. La defensa nº2: escala más desplazamiento (asimétrica)

Hasta ahora hemos supuesto que los pesos están centrados en 0 (escala **simétrica**: el rango va de
$-\max$ a $+\max$). Pero muchas magnitudes de un modelo no están centradas: por ejemplo, las salidas de
una función de activación como ReLU son **todas positivas**. Si guardas eso con una escala simétrica,
malgastas la mitad de los 256 niveles (los negativos) en valores que no existen. La cuantización
**asimétrica** lo arregla: usa una escala *y* un desplazamiento (un *zero-point*) para encajar el rango
real, sea cual sea. Lo medimos sobre datos solo positivos.


In [ ]:
activaciones = np.abs(np.random.randn(4096).astype(np.float32)) * 2.0   # solo positivas

def cuant_simetrica(w, bits=8):
    nivel = 2**(bits-1) - 1
    escala = np.abs(w).max() / nivel
    return np.clip(np.round(w/escala), -nivel, nivel) * escala

def cuant_asimetrica(w, bits=8):
    niveles = 2**bits - 1                                     # 255 niveles, de 0 a 255
    lo, hi = w.min(), w.max()
    escala = (hi - lo) / niveles
    q = np.clip(np.round((w - lo)/escala), 0, niveles)        # zero-point = lo
    return q*escala + lo

err_sim = np.abs(activaciones - cuant_simetrica(activaciones)).mean()
err_asim = np.abs(activaciones - cuant_asimetrica(activaciones)).mean()
print(f"Datos solo positivos (rango {activaciones.min():.2f} a {activaciones.max():.2f}):")
print(f"  error medio, cuantizacion simetrica : {err_sim:.6f}")
print(f"  error medio, cuantizacion asimetrica: {err_asim:.6f}")
print(f"\nLa asimetrica aprovecha los 256 niveles en el rango real: error x{err_sim/err_asim:.1f} menor.")


## 7. ¿Se nota al multiplicar? El efecto compuesto

Un modelo no usa los pesos sueltos: los multiplica por las entradas, miles de veces. Veamos si el error
de cuantización se acumula y estropea el resultado de una operación típica (multiplicar la matriz por un
vector), que es lo que hace una capa.


In [ ]:
x = np.random.randn(1024).astype(np.float32)
y_real = pesos @ x
y_cuant = (cuantiza(pesos, 8)[0].astype(np.float32) * cuantiza(pesos, 8)[1]) @ x
dif = np.abs(y_real - y_cuant)
print(f"Salida real vs cuantizada (8 bits) -> error medio {dif.mean():.4f} sobre valores de ~{np.abs(y_real).mean():.2f}")
print(f"Error relativo: {100*dif.mean()/np.abs(y_real).mean():.2f}%")
print("El error existe pero es pequeno frente a la senal: por eso la cuantizacion suele 'colar'.")


### El mismo experimento, bit a bit

¿Y si bajamos los bits en esa multiplicación? El error relativo de la salida es la cifra que de verdad
importa: es lo que el modelo «nota». Lo medimos a 8, 4, 3 y 2 bits para ver a partir de dónde la salida
deja de parecerse a la real.


In [ ]:
print("bits | error relativo de la salida")
for bits in [8, 4, 3, 2]:
    qb, eb = cuantiza(pesos, bits)
    y_b = (qb.astype(np.float32) * eb) @ x
    rel = 100 * np.abs(y_real - y_b).mean() / np.abs(y_real).mean()
    print(f"  {bits}  |        {rel:5.1f}%")
print("\nA 8 bits la salida apenas se mueve; a 4 bits ya se nota; a 3 y 2 bits se rompe del todo.")
print("(En modelos reales, Q4 aguanta mucho mejor que aqui: usan calibracion y escalas por bloque).")


## 8. Cuánto pesa un modelo de verdad

Cerramos con la cuenta que motiva todo esto: ¿cuánta memoria ocupa un modelo según los bits por peso?
Es una multiplicación, pero es la que decide si un modelo **cabe** en tu tarjeta gráfica o se queda
fuera. La regla mental: en *float16* son 2 bytes por parámetro; en 4 bits, medio byte.


In [ ]:
def memoria_gb(parametros, bits):
    return parametros * bits / 8 / 1e9

print(f"{'modelo':>14} | {'fp16 (16 bits)':>15} | {'int8 (8 bits)':>14} | {'Q4 (4 bits)':>12}")
for nombre, par in [("7.000 M", 7e9), ("13.000 M", 13e9), ("70.000 M", 70e9)]:
    print(f"{nombre:>14} | {memoria_gb(par,16):>12.1f} GB | {memoria_gb(par,8):>11.1f} GB | {memoria_gb(par,4):>9.1f} GB")
print("\nUn modelo de 13.000 M no cabe en una tarjeta de 16 GB en fp16, pero si en Q4.")
print("Esa diferencia (de no caber a caber en casa) es, en una linea, por que existe la cuantizacion.")


## 9. Pruébalo tú

1. **Cuantiza a 3 bits** y mira el error relativo de la salida (apartado 7). ¿En qué punto el resultado
   empieza a ser inservible? Ahí está el límite práctico.
2. **Sube la magnitud del *outlier*** en el barrido del apartado 4 (prueba 300, 1000): ¿hay tope para el
   daño? ¿Y si en vez de subir la magnitud metes diez atípicos del mismo tamaño?
3. **Compara las defensas:** ¿cuánto baja el error la escala por fila frente a la global en la matriz con
   varios *outliers*?
4. **Datos centrados frente a datos sesgados:** repite el apartado 6 con datos centrados en 0 (`np.random
   .randn`). ¿Sigue ganando la asimétrica o se igualan?
5. **Dimensiona tu hardware:** ¿cuántos GB necesita un modelo de 34.000 millones en Q4? ¿Cabe en tu
   tarjeta?


## 10. Errores comunes

- **Cuantizar sin mirar los *outliers*.** Un solo peso enorme degrada todos los demás. Lo has medido:
  estira la escala y arrastra a todos. Las técnicas serias los detectan y tratan aparte.
- **Bajar demasiado los bits.** A 2 bits casi todo se rompe; 4 bits es el filo habitual. No es gratis.
- **Olvidar que la escala importa.** Una escala por matriz es lo más barato pero lo menos preciso; por
  fila (o por bloque) es el estándar práctico.
- **Usar escala simétrica con datos sesgados.** Si tus números no están centrados en 0, malgastas la
  mitad de los niveles. La cuantización asimétrica (escala + desplazamiento) lo arregla.
- **Creer que cuantizar es siempre seguro.** En la mayoría de casos «cuela», pero en tareas muy
  sensibles puede degradar la calidad de forma notable: hay que medirlo, no asumirlo.


## 11. Qué te llevas

- **Cuantizar es redondear** los pesos a una rejilla con menos bits: 4× menos memoria al pasar de 32 a 8
  bits (y más aún a 4), con un error que casi siempre «cuela».
- El error **no es gratis**: con muy pocos bits o con *outliers* puede degradar el modelo. De ahí los
  trucos del oficio (por fila, asimétrica, *outliers* aparte, formatos como `Q4`).
- Lo has **medido** en todas sus caras: error por peso, señal/ruido, efecto de los atípicos, defensas y
  error compuesto al multiplicar.
- Es la razón técnica de que modelos enormes corran en hardware modesto (*edge AI*, capítulo 7): no se
  «encoge» el modelo, se **representa** más barato. La tabla de memoria lo deja claro: de no caber a
  caber en tu portátil.

**Para seguir:** el facsímil 4 te hace elegir entre nube y local sopesando precisamente esto —memoria,
latencia, coste y calidad—, y verás los formatos cuantizados (GGUF, GPTQ, AWQ) en acción.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*